Dataset Source: HUPD Patent Dataset (JSON format)

Initial Dataset Size:
31968 rows × 5 columns

Columns:
- application_number
- title
- abstract
- claims
- text

Text Column Creation:
text = title + abstract + claims

In [2]:
import pandas as pd
import numpy as np

In [4]:
import json
import os
from tqdm import tqdm

folder = "2018/2018"

data = []

files = os.listdir(folder)

for file in tqdm(files):
    
    if file.endswith(".json"):
        
        path = os.path.join(folder, file)
        
        with open(path, "r", encoding="utf-8") as f:
            
            patent = json.load(f)
            
            application_number = patent.get("application_number")
            title = patent.get("title", "")
            abstract = patent.get("abstract", "")
            claims = patent.get("claims", "")
            
            text = title + " " + abstract + " " + claims
            
            data.append({
                "application_number": application_number,
                "title": title,
                "abstract": abstract,
                "claims": claims,
                "text": text
            })

df = pd.DataFrame(data)

df.to_csv("patents_2018.csv", index=False)

print("CSV dataset created successfully")

100%|██████████| 31968/31968 [05:50<00:00, 91.15it/s] 


CSV dataset created successfully


In [5]:
df.tail()

,application_number,title,abstract,claims,text
31963,16062170,PYROTECHNICAL IGNITER,An electro-pyrotechnical igniter includes a me...,1. An electro-pyrotechnical igniter comprising...,PYROTECHNICAL IGNITER An electro-pyrotechnical...
31964,16062262,CAPSULE CUP OF A COFFEE CAPSULE,"A plastics-material capsule casing which, by v...",1. A rotationally symmetrical capsule cup made...,CAPSULE CUP OF A COFFEE CAPSULE A plastics-mat...
31965,16062675,"AIRBAG DEVICE FOR A MOTOR VEHICLE, AND AIRBAG ...","An airbag device for a motor vehicle, the airb...",1. An airbag device for a motor vehicle which ...,"AIRBAG DEVICE FOR A MOTOR VEHICLE, AND AIRBAG ..."
31966,16062981,LIPID FORMULATIONS CONTAINING BIOACTIVE FATTY ...,Provided herein is technology relating to comp...,1-32. (canceled) 33. A formulation comprising ...,LIPID FORMULATIONS CONTAINING BIOACTIVE FATTY ...
31967,16064534,THREE-DIMENSIONAL PROCESSING DEVICE,A three-dimensional processing device includes...,1-6. (canceled) 7. A three-dimensional process...,THREE-DIMENSIONAL PROCESSING DEVICE A three-di...


Reading the csv file created after converting json to csv

In [6]:
df = pd.read_csv("patents_2018.csv")
df.head()

,application_number,title,abstract,claims,text
0,13817165,Intelligent Drug and/or Fluid Delivery System ...,"A pharmacodynamic (PD), pharmacokinetic (PK), ...",1. A medication or fluid delivery and control ...,Intelligent Drug and/or Fluid Delivery System ...
1,14111139,ANTISTATIC DEVICE AND ASSOCIATED OPERATING METHOD,An antistatic device for reducing electrostati...,1. Antistatic device for reducing electrostati...,ANTISTATIC DEVICE AND ASSOCIATED OPERATING MET...
2,14112715,APPARATUS AND METHODS FOR ASSESSING THE EFFECT...,A method of assessing the effect of viewing va...,1. A method of assessing the effect of viewing...,APPARATUS AND METHODS FOR ASSESSING THE EFFECT...
3,14764986,"BLACK MATRIX, METHOD FOR MANUFACTURING THE SAM...","The disclosure provides a black matrix, a meth...","1. A black matrix, manufactured by the steps o...","BLACK MATRIX, METHOD FOR MANUFACTURING THE SAM..."
4,14765061,A LIGHTING CONTROL SYSTEM,The invention relates to the field of smart se...,"1. A lighting control system, applied for adju...",A LIGHTING CONTROL SYSTEM The invention relate...


In [7]:
df.shape

(31968, 5)

Data Cleaning Steps
1. Converted text to lowercase
2. Removed punctuation
3. Removed duplicate patent descriptions
4. Handled missing values

In [8]:
df["text"]=df["text"].str.lower()

In [9]:
import re
df["text"]=df["text"].apply(lambda x:re.sub(r"[^\w\s]","",x))

In [10]:
df=df.drop_duplicates(subset="text")

In [11]:
df.isnull().sum()

application_number    0
title                 0
abstract              0
claims                0
text                  0
dtype: int64

In [12]:
df.head()

,application_number,title,abstract,claims,text
0,13817165,Intelligent Drug and/or Fluid Delivery System ...,"A pharmacodynamic (PD), pharmacokinetic (PK), ...",1. A medication or fluid delivery and control ...,intelligent drug andor fluid delivery system t...
1,14111139,ANTISTATIC DEVICE AND ASSOCIATED OPERATING METHOD,An antistatic device for reducing electrostati...,1. Antistatic device for reducing electrostati...,antistatic device and associated operating met...
2,14112715,APPARATUS AND METHODS FOR ASSESSING THE EFFECT...,A method of assessing the effect of viewing va...,1. A method of assessing the effect of viewing...,apparatus and methods for assessing the effect...
3,14764986,"BLACK MATRIX, METHOD FOR MANUFACTURING THE SAM...","The disclosure provides a black matrix, a meth...","1. A black matrix, manufactured by the steps o...",black matrix method for manufacturing the same...
4,14765061,A LIGHTING CONTROL SYSTEM,The invention relates to the field of smart se...,"1. A lighting control system, applied for adju...",a lighting control system the invention relate...


Filtering the data set to get hardware designs

In [13]:
hardware_keywords = [
    "processor", "chip", "semiconductor", "circuit", "battery",
    "sensor", "transistor", "cooling", "thermal", "voltage",
    "current", "hardware", "device", "controller", "power",
    "signal", "microcontroller", "integrated circuit"
]
def is_hardware(text):
    for word in hardware_keywords:
        if word in text:
            return True
    return False

Applying hardware design filter to data set

In [16]:
df = df[df["text"].apply(is_hardware)]

Original size: 31968 rows
Sampled size: 400 rows

In [17]:
df = df.sample(400, random_state=42)

In [18]:
df.shape

(400, 5)

Creating a Data set csv file of reduced data

In [19]:
df.to_csv("reduuced_data_set.csv",index=False)

Importing TfidVectorizer from Sklearn through sklearn.feature_extraction.text

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

Specifying the features of to convert to vector

In [22]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2),
    token_pattern=r'[a-zA-Z]{2,}'
)

Applying the vector transformation

In [23]:
X = vectorizer.fit_transform(df["text"])

Shape of data set after transform

In [24]:
X.shape

(400, 5000)

Getting an example of the transformation

In [25]:
vectorizer.get_feature_names_out()[:20]

array(['aacmm', 'able', 'abnormal', 'abr', 'abutment', 'acceptable',
       'acceptable probability', 'access', 'access channel',
       'access operation', 'access point', 'access points',
       'access portion', 'access service', 'accessed', 'accessible',
       'accessing', 'accommodating', 'accommodation',
       'accommodation space'], dtype=object)

Feature matrix shape:
(400, 5000)

Meaning:
400 patent documents
5000 TF-IDF features

In [26]:
X.shape

(400, 5000)

X  → TF-IDF feature matrix
df["text"] → processed text dataset

In [27]:
tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_df.head(20)

,aacmm,able,abnormal,abr,abutment,acceptable,acceptable probability,access,access channel,access operation,...,ylpyridin ylmethylmethanamine,ylpyridin ylpentanamide,ylpyridin ylpivalamide,ylpyridin ylpropionamide,ylthiophen,ylthiophen ylethan,zinc,zone,zoom,zoom level
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.022862,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.031871,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
